In [ ]:
# --- ΒΗΜΑ 0: ΕΓΚΑΤΑΣΤΑΣΗ ΒΙΒΛΙΟΘΗΚΩΝ ---
print("⏳ Εγκατάσταση απαραίτητων βιβλιοθηκών...")
!pip install torch_geometric torch scikit-learn pandas numpy
print("✅ Η εγκατάσταση ολοκληρώθηκε.")

In [ ]:
# --- ΒΗΜΑ 1: ΦΟΡΤΩΣΗ & ΕΝΟΠΟΙΗΣΗ ---
import pandas as pd
import numpy as np

# 1. Φόρτωση αρχείων
try:
    df_pos = pd.read_csv('positive_protein_sequences.csv')
    df_neg = pd.read_csv('negative_protein_sequences.csv')

    # 2. Βάζουμε Ετικέτες (Labels): 1 = Σύνδεση, 0 = Καμία Σύνδεση
    df_pos['Label'] = 1
    df_neg['Label'] = 0

    # 3. Ένωση σε ένα μεγάλο πίνακα
    df = pd.concat([df_pos, df_neg], ignore_index=True)

    # Ανακάτεμα (Shuffle) για να μην είναι όλα τα 1 στην αρχή
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"✅ Επιτυχής φόρτωση!")
    print(f"Σύνολο δεδομένων: {len(df)} ζεύγη πρωτεϊνών.")
    print(f"Θετικά (Interactions): {len(df_pos)}")
    print(f"Αρνητικά (Non-interactions): {len(df_neg)}")

except FileNotFoundError:
    print("❌ ΣΦΑΛΜΑ: Δεν βρέθηκαν τα αρχεία csv. Ανέβασέ τα στο Colab!")

In [ ]:
# --- ΒΗΜΑ 2: BASELINE (RANDOM FOREST) ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import time

print("⏳ Προετοιμασία χαρακτηριστικών (k-mers) για το Random Forest...")

# 1. Συνάρτηση που σπάει την πρωτεΐνη σε τριπλέτες (π.χ. 'MKV' -> 'MKV', 'KVL'...)
def get_k_mers(sequence, k=3):
    return [sequence[i:i + k] for i in range(len(sequence) - k + 1)]

# Εφαρμογή στις στήλες
df['text_1'] = df['protein_sequences_1'].apply(lambda x: ' '.join(get_k_mers(x)))
df['text_2'] = df['protein_sequences_2'].apply(lambda x: ' '.join(get_k_mers(x)))

# 2. Μετατροπή σε αριθμούς (TF-IDF)
# Χρησιμοποιούμε max_features=1000 για ταχύτητα
vectorizer = TfidfVectorizer(max_features=1000)
all_text = pd.concat([df['text_1'], df['text_2']])
vectorizer.fit(all_text)

X1 = vectorizer.transform(df['text_1']).toarray()
X2 = vectorizer.transform(df['text_2']).toarray()

# Τα ενώνουμε: Τα χαρακτηριστικά του ζεύγους είναι X1 + X2
X = np.hstack((X1, X2))
y = df['Label'].values

# 3. Διαχωρισμός Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Εκπαίδευση
print("🚀 Ξεκινά η εκπαίδευση του Random Forest...")
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

# 5. Αξιολόγηση
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n📊 ΑΠΟΤΕΛΕΣΜΑΤΑ BASELINE:")
print(f"Accuracy: {acc*100:.2f}%")
print(f"F1-Score: {f1:.4f}")
print("------------------------------------------------")

In [ ]:
# --- ΒΗΜΑ 3: ΠΡΟΕΤΟΙΜΑΣΙΑ ΓΡΑΦΟΥ ΓΙΑ GNN ---
import torch
from torch_geometric.data import Data
import networkx as nx

print("⏳ Κατασκευή του Γράφου...")

# 1. Βρίσκουμε όλες τις μοναδικές πρωτεΐνες
unique_proteins = list(set(df['protein_sequences_1']).union(set(df['protein_sequences_2'])))
# Τους δίνουμε έναν αριθμό (ID): Πρωτεΐνη -> 0, 1, 2...
protein_to_id = {prot: i for i, prot in enumerate(unique_proteins)}

# 2. Φτιάχνουμε τις Ακμές (Edges) - ΜΟΝΟ ΑΠΟ ΤΑ ΘΕΤΙΚΑ ΔΕΙΓΜΑΤΑ
# Το GAT πρέπει να ξέρει ποια συνδέονται πραγματικά για να φτιάξει τη δομή
positive_pairs = df[df['Label'] == 1]
edges_src = [protein_to_id[p] for p in positive_pairs['protein_sequences_1']]
edges_dst = [protein_to_id[p] for p in positive_pairs['protein_sequences_2']] # FIXED: Changed from protein_sequences_1

# Το PyTorch θέλει τις ακμές σε μορφή tensor [2, num_edges]
edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)

# 3. Χαρακτηριστικά Κόμβων (Nodes)
# Εδώ κάνουμε το κόλπο: Δεν δίνουμε TF-IDF. Δίνουμε απλά IDs για να μάθει embeddings.
num_nodes = len(unique_proteins)
x = torch.arange(num_nodes) # Απλά οι αριθμοί 0, 1, 2...

# 4. Δημιουργία Αντικειμένου Data
data = Data(x=x, edge_index=edge_index)

print(f"✅ Ο Γράφος δημιουργήθηκε!")
print(f"Κόμβοι (Πρωτεΐνες): {data.num_nodes}")
print(f"Ακμές (Συνδέσεις): {data.num_edges}")

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
import random

# 1. ΡΥΘΜΙΣΗ ΓΙΑ ΣΤΑΘΕΡΑ ΑΠΟΤΕΛΕΣΜΑΤΑ (Seed)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

print("🔄 Φόρτωση και Επεξεργασία των δικών σου CSV...")

# 2. ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ
# Προσοχή: Άλλαξε τα ονόματα αρχείων αν χρειάζεται
df_pos = pd.read_csv('positive_protein_sequences.csv') # Πρέπει να έχει στήλες με ονόματα πρωτεϊνών
df_neg = pd.read_csv('negative_protein_sequences.csv')

# Αν οι στήλες σου έχουν άλλα ονόματα, άλλαξέ τα εδώ!
# Υποθέτω ότι η 1η στήλη είναι η Πρωτεΐνη Α και η 2η η Πρωτεΐνη Β
col_A = df_pos.columns[0]
col_B = df_pos.columns[1]

print(f"✅ Βρέθηκαν {len(df_pos)} θετικά και {len(df_neg)} αρνητικά ζεύγη.")

# 3. ΔΗΜΙΟΥΡΓΙΑ ΛΕΞΙΚΟΥ (MAPPING)
# Μαζεύουμε ΟΛΕΣ τις πρωτεΐνες (από θετικά και αρνητικά) για να φτιάξουμε τους κόμβους
all_proteins = set(df_pos[col_A]).union(set(df_pos[col_B])).union(set(df_neg[col_A])).union(set(df_neg[col_B]))
protein_to_idx = {prot: i for i, prot in enumerate(all_proteins)}
num_nodes = len(all_proteins)

print(f"🧬 Συνολικός Αριθμός Μοναδικών Πρωτεϊνών (Nodes): {num_nodes}")

# 4. ΜΕΤΑΤΡΟΠΗ ΣΕ TENSORS (Αριθμούς)
def process_edges(df):
    src = [protein_to_idx[p] for p in df[col_A]]
    dst = [protein_to_idx[p] for p in df[col_B]]
    return torch.tensor([src, dst], dtype=torch.long)

pos_edge_index = process_edges(df_pos)
neg_edge_index = process_edges(df_neg)

# 5. CUSTOM SPLIT (80% Train - 20% Test)
# Χωρίζουμε τα ΘΕΤΙΚΑ
pos_train_idx, pos_test_idx = train_test_split(range(pos_edge_index.shape[1]), test_size=0.2, random_state=42)
train_pos_edges = pos_edge_index[:, pos_train_idx]
test_pos_edges = pos_edge_index[:, pos_test_idx]

# Χωρίζουμε τα ΑΡΝΗΤΙΚΑ
neg_train_idx, neg_test_idx = train_test_split(range(neg_edge_index.shape[1]), test_size=0.2, random_state=42)
train_neg_edges = neg_edge_index[:, neg_train_idx]
test_neg_edges = neg_edge_index[:, neg_test_idx]

# 6. ΔΗΜΙΟΥΡΓΙΑ ΤΟΥ 'DATA' OBJECT (Η καρδιά του προβλήματος)
# Το 'edge_index' του γράφου θα έχει ΜΟΝΟ τα θετικά του training (για να περνάνε τα μηνύματα)
# Το 'edge_label_index' θα έχει τα ζεύγη που θέλουμε να προβλέψουμε (Train ή Test)

# --- Train Labels ---
train_edge_label_index = torch.cat([train_pos_edges, train_neg_edges], dim=1)
train_labels = torch.cat([torch.ones(train_pos_edges.shape[1]), torch.zeros(train_neg_edges.shape[1])], dim=0)

# --- Test Labels ---
test_edge_label_index = torch.cat([test_pos_edges, test_neg_edges], dim=1)
test_labels = torch.cat([torch.ones(test_pos_edges.shape[1]), torch.zeros(test_neg_edges.shape[1])], dim=0)

# Φτιάχνουμε το αντικείμενο
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


x_features = torch.arange(num_nodes).unsqueeze(1) # Dummy features (θα μάθει embeddings)

data = Data(x=x_features, edge_index=train_pos_edges) # Ο γράφος χτίζεται ΜΟΝΟ με τα γνωστά θετικά!
data = data.to(device)

# Στέλνουμε και τα labels στην GPU
train_edge_label_index = train_edge_label_index.to(device)
train_labels = train_labels.to(device)
test_edge_label_index = test_edge_label_index.to(device)
test_labels = test_labels.to(device)


# 7. ΤΟ ΜΟΝΤΕΛΟ GAT (ΙΔΙΟ ΜΕ ΠΡΙΝ)
class GATLinkPredictor(nn.Module):
    def __init__(self, num_nodes, embedding_dim, hidden_channels, heads=4):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.conv1 = GATConv(embedding_dim, hidden_channels, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False, dropout=0.3)

    def encode(self, x, edge_index):
        x = self.node_emb.weight # Χρησιμοποιούμε τα learnable embeddings
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        return (src * dst).sum(dim=-1)

model = GATLinkPredictor(num_nodes=num_nodes, embedding_dim=64, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

# 8. TRAINING LOOP (ΠΡΟΣΑΡΜΟΣΜΕΝΟ)
print("\n🚀 Ξεκινά η εκπαίδευση με CUSTOM NEGATIVES...")

def train():
    model.train()
    optimizer.zero_grad()

    # 1. Encode: Φτιάξε τα embeddings χρησιμοποιώντας τη δομή του Training Graph
    z = model.encode(data.x, data.edge_index)

    # 2. Decode: Προέβλεψε για τα ζεύγη του Training (Θετικά + Αρνητικά)
    out = model.decode(z, train_edge_label_index)

    # 3. Loss
    loss = criterion(out, train_labels)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    z = model.encode(data.x, data.edge_index)
    # Προέβλεψε για τα ζεύγη του Test (που δεν είδε στην εκπαίδευση)
    out = model.decode(z, test_edge_label_index)
    return roc_auc_score(test_labels.cpu().numpy(), out.sigmoid().cpu().numpy())

# Τρέξιμο
for epoch in range(1, 101):
    loss = train()
    if epoch % 10 == 0:
        val_auc = test() # Εδώ χρησιμοποιούμε το Test set ως Validation για ευκολία
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Test AUC: {val_auc:.4f}')

final_auc = test()
print(f"\n📊 ΤΕΛΙΚΟ ΑΠΟΤΕΛΕΣΜΑ (Hard Negatives): {final_auc:.4f}")

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

print("🔬 ΤΕΛΙΚΗ ΕΠΙΣΤΗΜΟΝΙΚΗ ΣΥΓΚΡΙΣΗ (ΚΑΙ ΤΑ 3 ΜΟΝΤΕΛΑ)")

# Συνάρτηση Metric Extraction
def get_detailed_metrics(embeddings, name):
    print(f"   ⏳ Analyzing {name}...")
    input_dim = embeddings.shape[1]

    # MLP Classifier (Ίδιος για όλους για δίκαιη σύγκριση)
    classifier = nn.Sequential(
        nn.Linear(input_dim * 2, 128),
        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 1)
    ).to(device)

    opt = torch.optim.Adam(classifier.parameters(), lr=0.005)
    loss_fn = nn.BCEWithLogitsLoss()

    # Train (Γρήγορη εκπαίδευση 50 εποχών)
    classifier.train()
    for _ in range(50):
        opt.zero_grad()
        src = embeddings[train_edge_label_index[0]]
        dst = embeddings[train_edge_label_index[1]]
        pair = torch.cat([src, dst], dim=1)
        loss = loss_fn(classifier(pair).squeeze(), train_labels)
        loss.backward()
        opt.step()

    # Evaluate
    classifier.eval()
    with torch.no_grad():
        src_t = embeddings[test_edge_label_index[0]]
        dst_t = embeddings[test_edge_label_index[1]]
        pair_t = torch.cat([src_t, dst_t], dim=1)

        logits = classifier(pair_t).squeeze()
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)
        y_true = test_labels.cpu().numpy()

        # Metrics
        acc = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds)
        prec = precision_score(y_true, preds)
        rec = recall_score(y_true, preds)
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

    return {
        "Model": name,
        "Accuracy": acc,
        "F1-Score": f1,
        "Precision": prec,
        "Recall": rec,
        "False Positives (FP)": fp
    }

# --- ΤΡΕΧΟΥΜΕ ΚΑΙ ΤΑ ΤΡΙΑ ---
# 1. ESM-2 (Χρησιμοποιούμε το aligned_esm_emb που είναι το σωστό)
if 'aligned_esm_emb' in locals():
    metrics_esm = get_detailed_metrics(aligned_esm_emb, "ESM-2 (Sequence)")
else:
    # Fallback αν χάθηκε η μεταβλητή (σπάνιο)
    metrics_esm = get_detailed_metrics(esm_emb, "ESM-2 (Sequence)")

# 2. GAT
metrics_gat = get_detailed_metrics(gat_emb, "GAT (Topology)")

# 3. Hybrid
metrics_hyb = get_detailed_metrics(hybrid_emb, "Hybrid (Fusion)")

# --- ΔΗΜΙΟΥΡΓΙΑ ΤΕΛΙΚΟΥ ΠΙΝΑΚΑ ---
df_results = pd.DataFrame([metrics_esm, metrics_gat, metrics_hyb])
df_results = df_results.set_index("Model")

print("\n📊 Ο ΠΙΝΑΚΑΣ ΤΗΣ ΔΙΠΛΩΜΑΤΙΚΗΣ:")
print(df_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
import torch
import torch.nn as nn
import numpy as np

# Ρυθμίσεις εμφάνισης
sns.set(style="whitegrid")
plt.rcParams.update({'font.size': 12})

print("🎨 Δημιουργία Βελτιωμένων Γραφημάτων (High Precision)...")

# Συνάρτηση που εκπαιδεύει ΠΙΟ ΠΡΟΣΕΚΤΙΚΑ για το γράφημα
def get_predictions_robust(embeddings, name):
    print(f"   ⏳ Training classifier for {name} plot...")
    input_dim = embeddings.shape[1]

    # Πιο "βαθύ" δίκτυο για να πιάσει τη λεπτομέρεια
    classifier = nn.Sequential(
        nn.Linear(input_dim * 2, 128),
        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 1)
    ).to(device)

    # 80 εποχές για σταθερή σύγκλιση
    opt = torch.optim.Adam(classifier.parameters(), lr=0.005)
    loss_fn = nn.BCEWithLogitsLoss()

    classifier.train()
    for epoch in range(80): # Αύξηση εποχών από 40 σε 80
        opt.zero_grad()
        src = embeddings[train_edge_label_index[0]]
        dst = embeddings[train_edge_label_index[1]]
        pair = torch.cat([src, dst], dim=1)
        loss = loss_fn(classifier(pair).squeeze(), train_labels)
        loss.backward()
        opt.step()

    classifier.eval()
    with torch.no_grad():
        src_t = embeddings[test_edge_label_index[0]]
        dst_t = embeddings[test_edge_label_index[1]]
        pair_t = torch.cat([src_t, dst_t], dim=1)
        logits = classifier(pair_t).squeeze()
        probs = torch.sigmoid(logits).cpu().numpy()

    return probs

# 1. Υπολογισμός Προβλέψεων
# Χρησιμοποιούμε τα embeddings που έχουμε ήδη
y_true = test_labels.cpu().numpy()

# Υπολογίζουμε ξανά με περισσότερες εποχές
probs_gat = get_predictions_robust(gat_emb, "GAT")
probs_hyb = get_predictions_robust(hybrid_emb, "Hybrid")

# Για το ESM (αν υπάρχει)
if 'aligned_esm_emb' in locals():
    probs_esm = get_predictions_robust(aligned_esm_emb, "ESM")
elif 'esm_emb' in locals():
    probs_esm = get_predictions_robust(esm_emb, "ESM")
else:
    probs_esm = None

# ==========================================
# ΓΡΑΦΗΜΑ 1: ROC CURVES (ZOOM IN)
# ==========================================
plt.figure(figsize=(10, 7))

# GAT
fpr_gat, tpr_gat, _ = roc_curve(y_true, probs_gat)
roc_auc_gat = auc(fpr_gat, tpr_gat)
plt.plot(fpr_gat, tpr_gat, color='blue', lw=2, label=f'GAT (AUC = {roc_auc_gat:.4f})')

# Hybrid
fpr_hyb, tpr_hyb, _ = roc_curve(y_true, probs_hyb)
roc_auc_hyb = auc(fpr_hyb, tpr_hyb)
# Κάνουμε τη γραμμή του Hybrid λίγο πιο παχιά για να φαίνεται
plt.plot(fpr_hyb, tpr_hyb, color='green', lw=3, linestyle='--', label=f'Hybrid (AUC = {roc_auc_hyb:.4f}) 🏆')

# ESM
if probs_esm is not None:
    fpr_esm, tpr_esm, _ = roc_curve(y_true, probs_esm)
    roc_auc_esm = auc(fpr_esm, tpr_esm)
    plt.plot(fpr_esm, tpr_esm, color='red', lw=2, alpha=0.5, label=f'ESM-2 (AUC = {roc_auc_esm:.4f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle=':')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Σύγκριση ROC Curves (Hybrid vs GAT vs ESM)')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

# ==========================================
# ΓΡΑΦΗΜΑ 2: MEIΩΣΗ ΛΑΘΩΝ (Bar Chart)
# ==========================================
# Φτιάχνουμε ένα Bar Chart που είναι πιο καθαρό για παρουσίαση
preds_gat = (probs_gat > 0.5).astype(int)
preds_hyb = (probs_hyb > 0.5).astype(int)
cm_gat = confusion_matrix(y_true, preds_gat)
cm_hyb = confusion_matrix(y_true, preds_hyb)
fp_gat = cm_gat[0, 1]
fp_hyb = cm_hyb[0, 1]

plt.figure(figsize=(8, 5))
models = ['GAT (Structure)', 'Hybrid (Structure + Seq)']
fps = [fp_gat, fp_hyb]
colors = ['#4c72b0', '#55a868'] # Μπλε και Πράσινο

bars = plt.bar(models, fps, color=colors, width=0.5)

# Γράφουμε τα νούμερα πάνω στις μπάρες
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height}', ha='center')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import numpy as np

print("🔄 ΑΝΑΚΤΗΣΗ GAT EMBEDDINGS & ΕΛΕΓΧΟΣ...")

# ---------------------------------------------------------
# ΒΗΜΑ 1: ΠΑΡΑΓΩΓΗ ΤΩΝ EMBEDDINGS (Εδώ λύνουμε το πρόβλημά σου)
# ---------------------------------------------------------
try:
    model.eval() # Βάζουμε το μοντέλο σε mode αξιολόγησης
    with torch.no_grad():
        # Ζητάμε από το μοντέλο να μας δώσει τα χαρακτηριστικά (encode)
        gat_emb = model.encode(data.x, data.edge_index)

    print(f"✅ Επιτυχία! Ανακτήθηκαν {gat_emb.shape[0]} embeddings.")

except NameError:
    print("❌ ΣΦΑΛΜΑ: Δεν βρέθηκε το 'model' ή το 'data' στη μνήμη.")
    print("👉 Λύση: Πρέπει να ξανατρέξεις το κελί της ΕΚΠΑΙΔΕΥΣΗΣ (Training) πιο πάνω.")
    raise SystemExit("Τρέξε πρώτα την εκπαίδευση και μετά έλα πάλι εδώ!")

# ---------------------------------------------------------
# ΒΗΜΑ 2: ΕΛΕΓΧΟΣ OVERFITTING (Learning Curves)
# ---------------------------------------------------------
print("🕵️ Ξεκινάει ο έλεγχος για Overfitting...")

# Φτιάχνουμε έναν προσωρινό "κριτή" (Classifier)
input_dim = gat_emb.shape[1]
classifier = nn.Sequential(
    nn.Linear(input_dim * 2, 128),
    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5), # Strong Dropout
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 1)
).to(device)

opt = torch.optim.Adam(classifier.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

train_scores = []
test_scores = []

# Τρέχουμε γρήγορα 40 εποχές για να δούμε τη συμπεριφορά
for epoch in range(40):
    classifier.train()
    opt.zero_grad()

    # Train Data
    src = gat_emb[train_edge_label_index[0]]
    dst = gat_emb[train_edge_label_index[1]]
    pair = torch.cat([src, dst], dim=1)

    out = classifier(pair).squeeze()
    loss = loss_fn(out, train_labels)
    loss.backward()
    opt.step()

    # Καταγραφή Scores
    with torch.no_grad():
        classifier.eval()
        # Train AUC
        train_auc = roc_auc_score(train_labels.cpu().numpy(), torch.sigmoid(out).cpu().numpy())

        # Test AUC
        src_t = gat_emb[test_edge_label_index[0]]
        dst_t = gat_emb[test_edge_label_index[1]]
        pair_t = torch.cat([src_t, dst_t], dim=1)
        test_auc = roc_auc_score(test_labels.cpu().numpy(), torch.sigmoid(classifier(pair_t).squeeze()).cpu().numpy())

    train_scores.append(train_auc)
    test_scores.append(test_auc)

# ---------------------------------------------------------
# ΒΗΜΑ 3: ΤΟ ΓΡΑΦΗΜΑ ΤΗΣ ΑΛΗΘΕΙΑΣ
# ---------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(train_scores, label='Train Score (Μαθαίνει)', color='blue', linewidth=2)
plt.plot(test_scores, label='Test Score (Γενικεύει)', color='green', linestyle='--', linewidth=2)

plt.title('GAT Check: Υπάρχει Overfitting;', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('AUC Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Αυτόματο Συμπέρασμα
final_gap = train_scores[-1] - test_scores[-1]
print(f"\n📊 Διαφορά Train-Test: {final_gap*100:.2f}%")

if final_gap < 0.05:
    print("✅ ΣΥΜΠΕΡΑΣΜΑ: ΟΧΙ, δεν έχουμε Overfitting!")
    print("   Οι γραμμές είναι κοντά, άρα το μοντέλο είναι αξιόπιστο.")
else:
    print("⚠️ ΠΡΟΣΟΧΗ: Οι γραμμές απέχουν. Ίσως χρειάζεται περισσότερο Dropout.")

In [ ]:
import numpy as np
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

print("🔄 ΑΝΑΚΤΗΣΗ ΑΡΧΕΙΩΝ & RANDOM FOREST...")

# ==============================================================================
# ΒΗΜΑ 1: ΦΟΡΤΩΣΗ ΑΠΟ ΤΑ ΑΡΧΕΙΑ .NPY (Η ΛΥΣΗ ΣΤΟ ΠΡΟΒΛΗΜΑ ΣΟΥ)
# ==============================================================================
print("   ... Φόρτωση esm2_embeddings_1.npy και esm2_embeddings_2.npy ...")

try:
    # Φορτώνουμε τα numpy arrays από τα αρχεία
    part1_np = np.load('esm2_embeddings_1.npy')
    part2_np = np.load('esm2_embeddings_2.npy')

    # Τα κάνουμε Tensors και τα στέλνουμε στην GPU (ή CPU)
    part1 = torch.tensor(part1_np).to(device)
    part2 = torch.tensor(part2_np).to(device)

    # Τα ενώνουμε
    aligned_esm_emb = torch.cat((part1, part2), dim=0)
    print(f"   ✅ ESM Embeddings φορτώθηκαν και ενώθηκαν! Shape: {aligned_esm_emb.shape}")

except FileNotFoundError:
    print("   ❌ ΣΦΑΛΜΑ: Δεν βρέθηκαν τα αρχεία .npy!")
    print("   👉 Ελέγξε στον φάκελο αριστερά αν υπάρχουν τα 'esm2_embeddings_1.npy'.")
    print("   Αν δεν υπάρχουν, πρέπει να τα ανεβάσεις ξανά ή να τρέξεις το ESM βήμα.")
    raise

# ==============================================================================
# ΒΗΜΑ 2: ΔΗΜΙΟΥΡΓΙΑ HYBRID EMBEDDING (GAT + ESM)
# ==============================================================================
# Βεβαιωνόμαστε ότι έχουμε και το GAT
if 'gat_emb' not in locals():
    print("   ... Ανάκτηση GAT embeddings από το μοντέλο...")
    model.eval()
    with torch.no_grad():
        gat_emb = model.encode(data.x, data.edge_index)

# Έλεγχος διαστάσεων (Safety Check)
min_len = min(gat_emb.shape[0], aligned_esm_emb.shape[0])
gat_emb = gat_emb[:min_len]
aligned_esm_emb = aligned_esm_emb[:min_len]

# Δημιουργία Hybrid
hybrid_emb = torch.cat((gat_emb, aligned_esm_emb), dim=1)
print(f"   ✅ Hybrid Embeddings έτοιμα: {hybrid_emb.shape}")

# ==============================================================================
# ΒΗΜΑ 3: ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ (TRAIN/TEST SPLIT)
# ==============================================================================
print("   ... Προετοιμασία δεδομένων για Random Forest...")

# Train Set
src_train = hybrid_emb[train_edge_label_index[0]]
dst_train = hybrid_emb[train_edge_label_index[1]]
X_train = torch.cat([src_train, dst_train], dim=1).cpu().detach().numpy()
y_train = train_labels.cpu().numpy()

# Test Set
src_test = hybrid_emb[test_edge_label_index[0]]
dst_test = hybrid_emb[test_edge_label_index[1]]
X_test = torch.cat([src_test, dst_test], dim=1).cpu().detach().numpy()
y_test = test_labels.cpu().numpy()

# ==============================================================================
# ΒΗΜΑ 4: ΕΚΠΑΙΔΕΥΣΗ RANDOM FOREST
# ==============================================================================
print("   🌲 Εκπαίδευση Random Forest (Λίγη υπομονή)...")
rf_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

# Αξιολόγηση
preds = rf_model.predict(X_test)
acc = accuracy_score(y_test, preds)

print(f"\n✅ Random Forest Accuracy: {acc*100:.2f}%")

# ==============================================================================
# ΒΗΜΑ 5: ΓΡΑΦΗΜΑ FEATURE IMPORTANCE
# ==============================================================================
importances = rf_model.feature_importances_

# Υπολογισμός ποσοστών
gat_dim = gat_emb.shape[1]
esm_dim = aligned_esm_emb.shape[1]
total_dim_per_node = gat_dim + esm_dim

gat_importance = np.sum(importances[0:gat_dim]) + np.sum(importances[total_dim_per_node : total_dim_per_node + gat_dim])
esm_importance = np.sum(importances[gat_dim : total_dim_per_node]) + np.sum(importances[total_dim_per_node + gat_dim :])

total = gat_importance + esm_importance
gat_pct = (gat_importance / total) * 100
esm_pct = (esm_importance / total) * 100

# Plot
plt.figure(figsize=(8, 5))
labels = ['GAT (Structure)', 'ESM-2 (Chemistry)']
values = [gat_pct, esm_pct]
colors = ['#1f77b4', '#d62728']

bars = plt.bar(labels, values, color=colors, alpha=0.85, width=0.6)
plt.ylabel('Σημαντικότητα (%)')
plt.title('Τι κοίταξε το Random Forest;')
plt.ylim(0, 110)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
             f'{height:.1f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')
plt.show()